# Chapter 10 · Skills

The procedures of Northside Veterinary Clinic (how to write a vaccine notice, how to book a surgery, what to do when someone cancels at short notice…) as skills: folders with a SKILL.md file, with a name, a description and the instructions. Four ways of giving them to the agent are compared: none, all in the system prompt, on demand (the model loads the one it needs) and chosen by the host, which looks for the skill most similar to the task.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. The clinic's skills

Each one is a folder with a SKILL.md.

In [ ]:
import os
for folder in ['adoption-checkup', 'late-cancellation', 'payment-reminder', 'pet-travel', 'surgery-booking', 'vaccine-estimate', 'vaccine-notice', 'weekly-report']:
    os.makedirs(f"skills/{folder}", exist_ok=True)

In [ ]:
%%writefile skills/adoption-checkup/SKILL.md
---
name: adoption-checkup
description: How to handle the first visit of a pet adopted from an animal shelter.
---

# First visit of an adopted pet

1. The first checkup is free if it is booked within a month of the adoption.
2. Book a 30-minute appointment with the reason "adoption checkup".
3. Ask the owner to bring the shelter's adoption papers and the vaccination record, if they have one.
4. At the visit, the vet checks the microchip and plans any missing vaccines and deworming.

In [ ]:
%%writefile skills/late-cancellation/SKILL.md
---
name: late-cancellation
description: What to do when canceling an appointment, especially with less than 24 hours' notice.
---

# Canceling an appointment

1. Find the appointment with `appointments_of` and cancel it with `cancel_appointment`.
2. If the appointment is less than 24 hours away (today or tomorrow), there is a $15 cancellation fee, unless it is canceled because of an emergency or the pet's illness.
3. If there is a fee, send the owner a notice with `notify_owner` saying that the appointment is canceled and that they will be charged $15, which is deducted from their next visit if they book it in the same month.
4. Answer the user saying that you canceled the appointment and whether there is a fee.

In [ ]:
%%writefile skills/payment-reminder/SKILL.md
---
name: payment-reminder
description: How to send a client a reminder of their unpaid invoices.
---

# Payment reminder

1. Look up the client's unpaid invoices with `accounts`.
2. Write a notice that lists each invoice with its number, its date and its amount, and the total owed.
3. If any invoice is more than 60 days old (today is 2026-10-20), add that no new appointments will be given, except for emergencies, until it is paid.
4. Send it with `notify_owner`, to the client's name.
5. Answer the user with the total and whether you added the 60-day warning.

In [ ]:
%%writefile skills/pet-travel/SKILL.md
---
name: pet-travel
description: What a pet needs to travel abroad.
---

# Traveling with a pet

1. To travel to the European Union, a pet needs a microchip, an EU health certificate and a rabies vaccine given at least 21 days before.
2. The certificate costs $20 and is issued at the visit.
3. Ireland, Finland, Malta and Norway require a tapeworm treatment (praziquantel) for dogs, 1 to 5 days before arrival.
4. Check with `vaccines` whether the rabies vaccine is up to date, and tell the owner what is missing.

In [ ]:
%%writefile skills/surgery-booking/SKILL.md
---
name: surgery-booking
description: How to book a surgery (neuter, extraction, soft tissue surgery) and prepare the owner.
---

# Booking a surgery

1. Surgeries are only booked in the morning, at 9:00, 9:30, 10:00, 10:30 or 11:00 a.m. Use `free_slots` to find the first of those times when the vet is free.
2. Book with `book_appointment`. The reason starts with "surgery:" followed by the operation, for example "surgery: neuter".
3. Look up the pet's species with `find_pet`.
4. Send the owner a notice with `notify_owner` that gives the day and the time, says that the pet must arrive at 8:30 a.m., and gives the fasting: 12 hours without food for dogs and cats; for rabbits, only 1 hour.
5. Answer the user with the booked appointment and the fasting you gave.

In [ ]:
%%writefile skills/vaccine-estimate/SKILL.md
---
name: vaccine-estimate
description: How to calculate the estimate for bringing a pet's vaccines up to date.
---

# Vaccine estimate

1. Look up the pet's vaccines with `vaccines` and keep the overdue ones.
2. Look up the price of each one with `price`.
3. Add up the prices.
4. Check with `pets_of` how many pets the owner has. If they have more than three, apply a 10% discount to the total.
5. Answer with the breakdown (each vaccine and its price), the discount if there is one, and the total with two decimals, in dollars.

In [ ]:
%%writefile skills/vaccine-notice/SKILL.md
---
name: vaccine-notice
description: How to write and send the overdue vaccines notice to a pet's owner.
---

# Overdue vaccines notice

1. Look up the pet's vaccines with `vaccines` and keep the ones that are overdue.
2. Look up the price of each one with `price` (for example, "rabies vaccine").
3. Write the notice like this:
   - Start with "Hello," followed by the owner's name.
   - Name each overdue vaccine, with the date it became overdue and its price in dollars.
   - End exactly with: "Call (503) 555-0100 to book an appointment. Northside Veterinary Clinic".
4. Send it with `notify_owner`.
5. Answer the user in one sentence: who you notified and about which vaccines.

In [ ]:
%%writefile skills/weekly-report/SKILL.md
---
name: weekly-report
description: How to prepare the weekly summary of a vet's schedule.
---

# A vet's weekly report

1. Use `free_slots` to look up the vet's free times from Monday to Friday.
2. Count, for each day, the appointments the vet has (the times that are not free).
3. Write a three-sentence summary: the day with the most appointments, the day with the most free slots and the total number of appointments in the week.

## 2. Load the model and the clinic

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0 sentence-transformers  # the versions the book was checked with

In [ ]:
%%writefile clinic_server.py
"""MCP server of Northside Veterinary Clinic: chapter 6's server, with the vaccine and
booking functions outside the server, so they can be reused.

It can be used in three ways: imported, with a client connected in memory; launched as
a process that talks over stdio; or launched as an HTTP service that several clients
connect to at once.
Usage: python clinic_server.py [data_folder] [--http PORT]
"""
import argparse
import copy
import json
import unicodedata
from datetime import date
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

TODAY = date(2026, 10, 20)                  # the date of the clinic's data


# --- The clinic's data -------------------------------------------------------

def load_data(folder="data"):
    """Read the clinic's files and leave the clinic as it was at the start."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES, VETS
    global TIMES

    def read(name):
        return json.loads((Path(folder) / name).read_text(encoding="utf-8"))

    PETS, OWNERS = read("pets.json"), read("owners.json")
    HISTORY, VACCINES = read("history.json"), read("vaccines.json")
    APPOINTMENTS0, PHARMACY = read("appointments.json"), read("pharmacy.json")
    INVOICES, VETS = read("invoices.json"), read("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()


def reset():
    """Leave the clinic as it was at the start: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without diacritics, to compare names however they are written."""
    text = unicodedata.normalize("NFD", str(text).lower())
    return "".join(c for c in text if not unicodedata.combining(c)).strip()


def age(birth_date):
    """Age in whole years on the date of the data."""
    b = date.fromisoformat(birth_date)
    return TODAY.year - b.year - ((TODAY.month, TODAY.day) < (b.month, b.day))


def pet_summary(pid):
    """The main data of a pet, with its owner's name."""
    p = PETS[pid]
    return {"name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": OWNERS[p["owner"]]["name"]}


def vet_of(name):
    """The id of a vet given by id or by name (or part of it), or None."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) \
                or strip_accents(name) in strip_accents(n):
            return v
    return None


def pet_of(name, owner=""):
    """The id of the only pet with that name (and owner, if given)."""
    ids = [p for p, d in PETS.items() if strip_accents(name) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        raise ToolError(f'There is no pet called "{name}"'
                        + (f" owned by {owner}" if owner else "") + " at the clinic.")
    if len(ids) > 1:
        raise ToolError(f'There are {len(ids)} pets called "{name}"; give the '
                        "owner's name.")
    return ids[0]


def free(date_, vet=""):
    """The free times of a weekday, for one vet or for all of them."""
    try:
        day = date.fromisoformat(date_)
    except ValueError:
        raise ToolError(f'"{date_}" is not a date; use YYYY-MM-DD.')
    if day.weekday() >= 5:
        raise ToolError(f"{date_} is a weekend day and the clinic is closed.")
    vid = vet_of(vet) if vet else None
    if vet and vid is None:
        raise ToolError(f'There is no vet "{vet}". They are: '
                        + ", ".join(VETS.values()) + ".")
    taken = {(a["vet"], a["time"]) for a in APPOINTMENTS
             if a["date"] == date_ and a["status"] != "canceled"}
    return {"date": date_, "free_times": {
        VETS[v]: [t for t in TIMES if (v, t) not in taken]
        for v in ([vid] if vid else VETS)}}


def vaccines_of(pet, owner=""):
    """A pet's vaccines, each with its next date and whether it is overdue."""
    pid = pet_of(pet, owner)
    return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "vaccines": [
        {"vaccine": v["vaccine"], "next": v["next"],
         "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
        for v in VACCINES[pid]]}


def book_appointment_data(pet, vet, date_, time, reason, owner=""):
    """Book an appointment, if all the data are valid and the time is free."""
    pid, vid = pet_of(pet, owner), vet_of(vet)
    if vid is None:
        raise ToolError(f'There is no vet "{vet}".')
    free(date_, vet)
    if time not in TIMES:
        raise ToolError(f'"{time}" is not an appointment time.')
    if any(a["date"] == date_ and a["time"] == time and a["vet"] == vid
           and a["status"] != "canceled" for a in APPOINTMENTS):
        raise ToolError(f"{VETS[vid]} already has an appointment on {date_} at {time}.")
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date_, "time": time, "vet": vid, "pet": pid,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return {"booked": appointment["id"], "pet": PETS[pid]["name"], "vet": VETS[vid],
            "date": date_, "time": time}


# --- The server --------------------------------------------------------------

Name = Annotated[str, Field(description="Name of the pet.")]
Owner = Annotated[str, Field(
    description="Owner's name; only if several pets have that name.")]
Date = Annotated[str, Field(description="YYYY-MM-DD.")]


def create_server():
    """The clinic's server, with its eight tools."""
    server = MCPServer("clinic", instructions="Pets, vaccines, medical history, "
                       "schedule, notices and invoices of Northside Veterinary Clinic.")

    @server.tool()
    def find_pet(name: Name, owner: Owner = "") -> dict:
        """Data of a pet looked up by its name: species, age, weight, allergies and
        owner."""
        return pet_summary(pet_of(name, owner))

    @server.tool()
    def vaccines(pet: Name, owner: Owner = "") -> dict:
        """A pet's vaccines: the next date of each one and whether it is overdue."""
        return vaccines_of(pet, owner)

    @server.tool()
    def history(pet: Name, last: Annotated[int, Field(
            description="How many visits, from the most recent to the oldest.")] = 3,
                owner: Owner = "") -> dict:
        """A pet's last visits, starting with the most recent."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "visits": [
            {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
            for v in reversed(HISTORY[pid][-last:])]}

    @server.tool()
    def free_slots(date: Date, vet: Annotated[str, Field(
            description="Name of the vet; empty for all of them.")] = "") -> dict:
        """Free times in the schedule on a weekday, for one vet or for all of them."""
        return free(date, vet)

    @server.tool()
    def appointments_of(pet: Name, owner: Owner = "") -> dict:
        """A pet's appointments (past and scheduled), with their id."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "appointments": [
            {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
             "vet": VETS[a["vet"]], "status": a["status"]}
            for a in APPOINTMENTS if a["pet"] == pid]}

    @server.tool()
    def book_appointment(pet: Name,
                         vet: Annotated[str, Field(description="Name of the vet.")],
                         date: Date, time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
        """Book an appointment at a free time."""
        return book_appointment_data(pet, vet, date, time, reason, owner)

    @server.tool()
    def cancel_appointment(appointment: Annotated[str, Field(
            description="Id of the appointment, such as AP-0026 (given by "
                        "appointments_of).")]) -> dict:
        """Cancel a scheduled appointment."""
        for a in APPOINTMENTS:
            if a["id"] == appointment:
                if a["status"] != "scheduled":
                    raise ToolError(f'Appointment {appointment} is "{a["status"]}".')
                a["status"] = "canceled"
                return {"canceled": appointment, "date": a["date"], "time": a["time"]}
        raise ToolError(f"Appointment {appointment} does not exist.")

    @server.tool()
    def upcoming_appointments() -> dict:
        """All the scheduled appointments in the schedule, to check what was booked."""
        return {"appointments": [{k: a[k] for k in ("id", "date", "time", "vet", "pet")}
                                 for a in APPOINTMENTS if a["status"] == "scheduled"]}

    return server


if __name__ == "__main__":
    arguments = argparse.ArgumentParser()
    arguments.add_argument("data", nargs="?", default="data")
    arguments.add_argument("--http", type=int,
                           help="port; without it, the server talks over stdio")
    a = arguments.parse_args()
    load_data(a.data)
    server = create_server()
    if a.http:
        server.run(transport="streamable-http", host="127.0.0.1", port=a.http)
    else:
        server.run()

In [ ]:
"""Chapter 10: skills. The clinic's procedures, first in the system prompt and then in
folders with a SKILL.md that the agent loads when it needs them."""
import asyncio
import json
import re
import sys
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the environment
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


import clinic_server


def load(name=None):
    """Load the model and its tokenizer: on the GPU, in float16; otherwise, on the
    CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))


def fetch_data():
    """The clinic's files, in the data folder; in Colab, they are downloaded."""
    DATA.mkdir(exist_ok=True)
    for name in ("pets", "owners", "history", "vaccines", "appointments", "pharmacy",
                 "invoices", "vets", "prices"):
        path = DATA / f"{name}.json"
        if not path.exists():
            urllib.request.urlretrieve(REPOSITORY + "data/" + path.name, path)

In [ ]:
tok, model = load(MODEL)
fetch_data()
clinic_server.load_data(DATA)

In [ ]:
SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer of the model, with its input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds and the model does not
    need."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """The JSON of a result on a single line, without the indentation the server
    sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 4's loop, with the host's extra text in the system prompt."""
    messages = [{"role": "system", "content": SYSTEM + host.extra},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                arguments = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({arguments})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
def create_server():
    """Chapter 7's server with four more tools: notices, accounts, prices and a
    client's pets."""
    server = clinic_server.create_server()
    cs = clinic_server
    prices = download("prices.json")

    @server.tool()
    def notify_owner(recipient: Annotated[str, Field(
                         description="Name of the client, or of their pet.")],
                     text: Annotated[str, Field(
                         description="Text of the notice, in English.")]) -> dict:
        """Send a notice to a client, by email or SMS as they prefer."""
        clients = [c for c, x in cs.OWNERS.items()
                   if cs.strip_accents(recipient) == cs.strip_accents(x["name"])]
        owner = clients[0] if clients else cs.PETS[cs.pet_of(recipient)]["owner"]
        cs.NOTICES.append({"owner": owner, "text": text})
        return {"sent": True, "to": cs.OWNERS[owner]["name"],
                "via": cs.OWNERS[owner]["notify_by"]}

    @server.tool()
    def accounts(owner: Annotated[str, Field(description="Name of the client.")]
                 ) -> dict:
        """A client's unpaid invoices, with their date and amount, and the total."""
        ids = [c for c, x in cs.OWNERS.items()
               if cs.strip_accents(owner) in cs.strip_accents(x["name"])]
        if not ids:
            raise ToolError(f'There is no client called "{owner}".')
        unpaid = [i for i in cs.INVOICES if i["owner"] in ids and not i["paid"]]
        return {"client": cs.OWNERS[ids[0]]["name"],
                "invoices": [{"id": i["id"], "date": i["date"], "total": i["total"]}
                             for i in unpaid],
                "total": round(sum(i["total"] for i in unpaid), 2)}

    @server.tool()
    def price(service: Annotated[str, Field(
            description='Service, such as "rabies vaccine".')]) -> dict:
        """Price in dollars of a clinic service."""
        words = set(cs.strip_accents(service).split())
        best = {k: v for k, v in prices.items()
                if words & set(cs.strip_accents(k).split()) - {"vaccine"}}
        if not best:
            raise ToolError(f'There is no price for "{service}". Services: '
                            + ", ".join(prices) + ".")
        return {"prices_usd": best}

    @server.tool()
    def pets_of(owner: Annotated[str, Field(description="Name of the client.")]
                ) -> dict:
        """A client's pets."""
        ids = [c for c, x in cs.OWNERS.items()
               if cs.strip_accents(owner) in cs.strip_accents(x["name"])]
        return {"pets": [{"name": p["name"], "species": p["species"]}
                         for p in cs.PETS.values() if p["owner"] in ids]}

    return server

## 3. Read the skills and give them to the agent

In [ ]:
SKILLS = Path("skills")                    # in the notebook, in the working folder


EMBEDDER = "intfloat/multilingual-e5-small"


def read_skills(folder=SKILLS):
    """Each skill is a folder with a SKILL.md: a header with its name and its
    description between two "---" lines, and the instructions below."""
    skills = {}
    for path in sorted(folder.glob("*/SKILL.md")):
        _, header, body = path.read_text(encoding="utf-8").split("---", 2)
        data = dict(line.split(": ", 1) for line in header.strip().splitlines())
        skills[data["name"]] = {"description": data["description"],
                                "instructions": body.strip()}
    return skills

In [ ]:
EMBEDDINGS = SentenceTransformer(EMBEDDER, device="cpu")     # on the CPU: the GPU is the language model's

In [ ]:
skills = read_skills()
for name, s in skills.items():
    n = len(tok(s["instructions"])["input_ids"])
    print(f"{name:22s} {n:4d} tokens  {s['description']}")

In [ ]:
THRESHOLD = 0.82       # minimum similarity between the task and a skill's description


class Host:
    """The compact host, with the skills in four modes: none, all in the system
    prompt, on demand (the model sees the list and loads the one it needs with
    load_skill) or chosen by the host (the skill whose description is most similar to
    the task, if it is similar enough, according to an embedding model)."""

    def __init__(self, server, skills, mode="none"):
        self.server, self.skills, self.mode = server, skills, mode
        self.loaded = []

    def choose(self, task):
        """In the "the host" mode, put the skill that fits in the system prompt."""
        if self.mode != "the host":
            return
        names = list(self.skills)
        descriptions = EMBEDDINGS.encode(
            [f"passage: {n}: {self.skills[n]['description']}" for n in names],
            normalize_embeddings=True, show_progress_bar=False)
        query = EMBEDDINGS.encode([f"query: {task}"], normalize_embeddings=True,
                                  show_progress_bar=False)[0]
        similarities = descriptions @ query
        best = int(similarities.argmax())
        if similarities[best] >= THRESHOLD:
            self.loaded.append(names[best])
            self.extra = ("\n\nProcedure to follow:\n\n"
                          + self.skills[names[best]]["instructions"])

    async def __aenter__(self):
        self.client = Client(self.server)
        await self.client.__aenter__()
        self.schemas = [{"type": "function", "function": {
            "name": t.name, "description": t.description,
            "parameters": without_titles(t.input_schema)}}
            for t in (await self.client.list_tools()).tools]
        self.extra = ""
        if self.mode == "all":
            self.extra = "\n\nProcedures of the clinic:\n\n" + "\n\n".join(
                s["instructions"] for s in self.skills.values())
        elif self.mode == "on demand":
            self.extra = ("\n\nProcedures of the clinic (load the one that fits the "
                          "task with load_skill before starting):\n" + "\n".join(
                              f"- {n}: {s['description']}"
                              for n, s in self.skills.items()))
            self.schemas.append({"type": "function", "function": {
                "name": "load_skill",
                "description":
                    "Load the instructions of one of the clinic's procedures.",
                "parameters": {"type": "object", "required": ["name"], "properties": {
                    "name": {"type": "string", "enum": list(self.skills)}}}}})
        return self

    async def __aexit__(self, *error):
        await self.client.__aexit__(*error)

    async def call(self, name, arguments):
        """Run a call: load_skill here, and any other tool on the server."""
        if name == "load_skill":
            skill = self.skills.get(arguments.get("name"))
            if skill is None:
                return json.dumps({"error": "There is no such procedure."})
            self.loaded.append(arguments["name"])
            return skill["instructions"]
        r = await self.client.call_tool(name, arguments)
        return compact("\n".join(b.text for b in r.content if b.type == "text"))

## 4. Six tasks and four modes

In [ ]:
def notice_to(name, *forms):
    """Is there a notice for that client with all those texts?"""
    cs = clinic_server
    return any(cs.OWNERS[n["owner"]]["name"] == name
               and all(re.search(f, n["text"], re.I) for f in forms)
               for n in cs.NOTICES)


def new_appointments():
    """The appointments booked since the clinic was reset."""
    cs = clinic_server
    old = {x["id"] for x in cs.APPOINTMENTS0}
    return [a for a in cs.APPOINTMENTS if a["id"] not in old]


def snowballs_surgery(answer):
    """One surgery at 09:00 with Dr. Fenton, and the owner told the arrival and the
    fasting."""
    booked = new_appointments()
    return (len(booked) == 1 and booked[0]["time"] == "09:00"
            and booked[0]["vet"] == "V3"
            and booked[0]["reason"].lower().startswith("surg")
            and notice_to("Peter Adams",
                          r"\b1[- ]hour|\bone[- ]hour|(?<!half )\ban hour", r"8[:.]30"))


TASKS = [
    ("toby_notice", "Let Toby's owner know about his overdue vaccines.",
     "vaccine-notice",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis", r"\b30\b",
                         r"\b38\b", r"\b45\b", r"\(503\) 555-0100", "Northside")),
    ("toby_estimate", "Prepare the estimate to bring Toby's vaccines up to date.",
     "vaccine-estimate", lambda r: says(r, r"\b113\.00\b")),
    ("snowball_surgery",
     "Book a neuter for Snowball with Dr. Lucy Fenton on Monday the 26th.",
     "surgery-booking", snowballs_surgery),
    ("cancel_rocky", "Cancel Rocky's appointment tomorrow.", "late-cancellation",
     lambda r: any(a["id"] == "AP-0060" and a["status"] == "canceled"
                   for a in clinic_server.APPOINTMENTS)
     and notice_to("Carol Roberts", r"\b15\b")),
    ("paula_reminder", "Send a payment reminder to Paula Ingram.", "payment-reminder",
     lambda r: notice_to("Paula Ingram", "INV-0016", r"\b207\b", "appointment")),
    ("lola_rabies", "When is Lola due for her rabies vaccine?", None,
     lambda r: says(r, "2027") and says(r, r"\baug", "-08-", r"\b0?8/\d")),
]


async def measure(mode, skills):
    """The six tasks with one way of giving the skills to the agent."""
    rows = []
    for key, task, skill, check in TASKS:
        clinic_server.reset()
        async with Host(create_server(), skills, mode) as host:
            host.choose(task)
            answer, trace = await mcp_agent(task, host)
            loaded = host.loaded
        rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                     "skill": skill, "loaded": loaded,
                     "notices": list(clinic_server.NOTICES),
                     "input_tokens": sum(s["input_tokens"] for s in trace),
                     "steps": len(trace),
                     "calls": [(c["name"], c["arguments"]) for s in trace
                               for c in s["calls"]]})
    r = {"ok": sum(w["ok"] for w in rows),
         "input_tokens": sum(w["input_tokens"] for w in rows),
         "steps": sum(w["steps"] for w in rows),
         "right_skill": sum(w["skill"] in w["loaded"] for w in rows if w["skill"]),
         "rows": rows}
    print(f"skills {mode:10s}", {k: v for k, v in r.items() if k != "rows"}, flush=True)
    print("    " + " ".join(("+" if w["ok"] else "-") + w["task"] for w in rows),
          flush=True)
    return r

In [ ]:
results = {mode: await measure(mode, skills)
           for mode in ("none", "all", "on demand", "the host")}

## Exercises

1. Explain in your own words the difference between a skill, an MCP prompt and a document as a resource.

2. Count the tokens of the list of skills and those of all the instructions. With how many skills of that size does everything stop fitting in the system prompt of a model whose context, the maximum number of tokens it can read at once, is 32,000?

3. Change the description of `surgery-booking` to a vague one, such as "Operating room procedure." Does the host still choose it when asked for a neuter?

4. Compute the similarity of each task with each skill and count in how many tasks the host chooses right with thresholds of 0.78, 0.80, 0.82, 0.84 and 0.86. What happens above and below?

5. Make the instruction of the on-demand mode more forceful: it should say that, if a skill fits, it must be loaded with the first call. Does the model load the skills now? How many tasks does it do right?

6. While `late-cancellation` is loaded, make the host show only three tools: `appointments_of`, `cancel_appointment` and `notify_owner`. Does anything change in Rocky's task?

7. Fix `vaccine-notice` in three ways: insisting in step 4 that the notice is sent with `notify_owner` and not written in the answer; adding a step that looks up the pet with `find_pet` to learn the owner's name; and, in addition, asking for the call to `notify_owner` in the same step that says how to write the notice. With which one does the agent send the notice?

8. Look in the results at which skills the agent had in front of it in each task with the "on demand" and "the host" modes. Was there any it did not need?

9. Think about security: a third-party skill is text that the model obeys. Which defenses of chapter 9 would also work for skills?

10. **Project.** Write three skills for your project from chapter 4 and measure the agent without them, with all of them in the system prompt and chosen by the host.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills) (Anthropic, 2025) and its [open specification](https://agentskills.io). Northside Veterinary Clinic, its data and its procedures are fictional. Original code and experiments.